In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.metrics import (
    confusion_matrix, roc_curve, auc,
    precision_recall_curve, average_precision_score
)

plt.rcParams.update({
    'figure.dpi'        : 120,
    'axes.spines.top'   : False,
    'axes.spines.right' : False,
    'axes.titlesize'    : 13,
    'axes.titleweight'  : 'bold',
    'axes.labelsize'    : 11,
    'xtick.labelsize'   : 10,
    'ytick.labelsize'   : 10,
    'font.family'       : 'sans-serif'
})

CHURN_COLOR  = '#E8593C'
RETAIN_COLOR = '#3B8BD4'
LR_COLOR     = '#7F77DD'   # purple  = logistic regression
RF_COLOR     = '#1D9E75'   # teal    = random forest
RF_T_COLOR   = '#E8593C'   # coral   = tuned RF

os.makedirs('../outputs', exist_ok=True)

# ── Load everything ───────────────────────────────────────────────────────────
y_test        = joblib.load('../outputs/y_test.pkl')
X_test        = joblib.load('../outputs/X_test.pkl')
feature_names = joblib.load('../outputs/feature_names.pkl')
preds         = joblib.load('../outputs/predictions.pkl')
results_df    = joblib.load('../outputs/model_comparison.pkl')
rf_tuned      = joblib.load('../models/random_forest_tuned.pkl')
lr_model      = joblib.load('../models/logistic_regression.pkl')

lr_pred        = preds['lr_pred'];       lr_proba        = preds['lr_proba']
rf_pred        = preds['rf_pred'];       rf_proba        = preds['rf_proba']
rf_tuned_pred  = preds['rf_tuned_pred']; rf_tuned_proba  = preds['rf_tuned_proba']

print('All data loaded ✓')

In [ ]:
# ── Chart 1: Model comparison bar chart ──────────────────────────────────────
metrics_to_plot = ['Accuracy', 'Precision', 'Recall', 'F1', 'ROC-AUC']
x      = np.arange(len(metrics_to_plot))
width  = 0.25
models = results_df.index.tolist()
colors = [LR_COLOR, RF_COLOR, RF_T_COLOR]

fig, ax = plt.subplots(figsize=(12, 5))
for i, (model, color) in enumerate(zip(models, colors)):
    vals = results_df.loc[model, metrics_to_plot].values
    bars = ax.bar(x + i*width, vals, width, label=model, color=color,
                  edgecolor='white', alpha=0.9)
    for bar in bars:
        ax.text(
            bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.005,
            f'{bar.get_height():.3f}',
            ha='center', va='bottom', fontsize=7.5
        )

ax.set_xticks(x + width)
ax.set_xticklabels(metrics_to_plot)
ax.set_ylim(0, 1.12)
ax.set_ylabel('Score')
ax.set_title('Model Performance Comparison')
ax.legend(loc='lower right', fontsize=9)
ax.yaxis.set_major_formatter(mtick.FormatStrFormatter('%.2f'))
ax.axhline(0.8, color='gray', linestyle='--', linewidth=0.8, alpha=0.5, label='0.80 threshold')

plt.tight_layout()
plt.savefig('../outputs/eval_01_model_comparison.png', bbox_inches='tight')
plt.show()
print('Chart 01 saved ✓')

In [ ]:
# ── Chart 2: Confusion matrices (all 3 models) ───────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

model_preds = [
    ('Logistic Regression', lr_pred,       LR_COLOR),
    ('Random Forest',       rf_pred,       RF_COLOR),
    ('Random Forest (tuned)', rf_tuned_pred, RF_T_COLOR),
]

for ax, (name, pred, color) in zip(axes, model_preds):
    cm = confusion_matrix(y_test, pred)
    sns.heatmap(
        cm, annot=True, fmt='d',
        cmap=sns.light_palette(color, as_cmap=True),
        xticklabels=['Retained', 'Churned'],
        yticklabels=['Retained', 'Churned'],
        linewidths=0.5,
        ax=ax, cbar=False
    )
    ax.set_title(name)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual' if ax == axes[0] else '')

    tn, fp, fn, tp = cm.ravel()
    ax.set_xlabel(f'Predicted\n(TP={tp}, FP={fp}, TN={tn}, FN={fn})', fontsize=9)

plt.suptitle('Confusion Matrices', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../outputs/eval_02_confusion_matrices.png', bbox_inches='tight')
plt.show()
print('Chart 02 saved ✓')

In [ ]:
# ── Chart 3: ROC curves ───────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 6))

roc_models = [
    ('Logistic Regression',   lr_proba,       LR_COLOR),
    ('Random Forest',         rf_proba,       RF_COLOR),
    ('Random Forest (tuned)', rf_tuned_proba, RF_T_COLOR),
]

for name, proba, color in roc_models:
    fpr, tpr, _ = roc_curve(y_test, proba)
    roc_auc     = auc(fpr, tpr)
    ax.plot(fpr, tpr, color=color, linewidth=2, label=f'{name}  (AUC = {roc_auc:.3f})')

ax.plot([0, 1], [0, 1], 'k--', linewidth=1, alpha=0.4, label='Random classifier (AUC = 0.500)')
ax.fill_between([0, 1], [0, 1], alpha=0.03, color='gray')

ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves — All Models')
ax.legend(loc='lower right', fontsize=9)
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.02])

plt.tight_layout()
plt.savefig('../outputs/eval_03_roc_curves.png', bbox_inches='tight')
plt.show()
print('Chart 03 saved ✓')

In [ ]:
# ── Chart 4: Precision-Recall curves ─────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 6))

for name, proba, color in roc_models:
    precision, recall, _ = precision_recall_curve(y_test, proba)
    ap = average_precision_score(y_test, proba)
    ax.plot(recall, precision, color=color, linewidth=2, label=f'{name}  (AP = {ap:.3f})')

baseline = y_test.mean()
ax.axhline(baseline, color='gray', linestyle='--', linewidth=1,
           alpha=0.6, label=f'Baseline (random) = {baseline:.2f}')

ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('Precision-Recall Curves')
ax.legend(loc='upper right', fontsize=9)
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.02])

plt.tight_layout()
plt.savefig('../outputs/eval_04_precision_recall.png', bbox_inches='tight')
plt.show()
print('Chart 04 saved ✓')

In [ ]:
# ── Chart 5: Feature importance — Random Forest (tuned) ──────────────────────
importances = rf_tuned.feature_importances_
feat_df = (
    pd.DataFrame({'Feature': feature_names, 'Importance': importances})
    .sort_values('Importance', ascending=False)
    .head(20)
)

fig, ax = plt.subplots(figsize=(9, 7))
bars = ax.barh(
    feat_df['Feature'][::-1],
    feat_df['Importance'][::-1],
    color=RF_T_COLOR, edgecolor='white', alpha=0.85
)
for bar in bars:
    ax.text(
        bar.get_width() + 0.001,
        bar.get_y() + bar.get_height()/2,
        f'{bar.get_width():.3f}',
        va='center', fontsize=8
    )
ax.set_xlabel('Feature Importance (Gini)')
ax.set_title('Top 20 Features — Random Forest (tuned)')
ax.set_xlim(0, feat_df['Importance'].max() * 1.18)

plt.tight_layout()
plt.savefig('../outputs/eval_05_feature_importance.png', bbox_inches='tight')
plt.show()

print('Top 5 most important features:')
for i, (_, row) in enumerate(feat_df.head(5).iterrows(), 1):
    print(f'  {i}. {row["Feature"]:<40} {row["Importance"]:.4f}')
print('Chart 05 saved ✓')

In [ ]:
# ── Chart 6: Logistic Regression coefficients ─────────────────────────────────
coef_df = (
    pd.DataFrame({'Feature': feature_names, 'Coefficient': lr_model.coef_[0]})
    .assign(Abs=lambda d: d['Coefficient'].abs())
    .sort_values('Abs', ascending=False)
    .head(20)
    .sort_values('Coefficient')
)

fig, ax = plt.subplots(figsize=(9, 7))
colors_coef = [CHURN_COLOR if v > 0 else RETAIN_COLOR for v in coef_df['Coefficient']]
bars = ax.barh(coef_df['Feature'], coef_df['Coefficient'],
               color=colors_coef, edgecolor='white', alpha=0.85)
ax.axvline(0, color='gray', linewidth=0.8)
ax.set_xlabel('Coefficient value')
ax.set_title('Logistic Regression — Top 20 Feature Coefficients')

from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor=CHURN_COLOR,  label='Increases churn risk'),
    Patch(facecolor=RETAIN_COLOR, label='Decreases churn risk'),
]
ax.legend(handles=legend_elements, loc='lower right', fontsize=9)

plt.tight_layout()
plt.savefig('../outputs/eval_06_lr_coefficients.png', bbox_inches='tight')
plt.show()
print('Chart 06 saved ✓')

In [ ]:
# ── Chart 7: Prediction probability distribution ─────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, (name, proba, color) in zip(axes, [
    ('Logistic Regression',   lr_proba,       LR_COLOR),
    ('Random Forest (tuned)', rf_tuned_proba, RF_T_COLOR),
]):
    actual_0 = proba[y_test == 0]
    actual_1 = proba[y_test == 1]

    ax.hist(actual_0, bins=30, alpha=0.6, color=RETAIN_COLOR, label='Actual: Retained', edgecolor='white')
    ax.hist(actual_1, bins=30, alpha=0.7, color=CHURN_COLOR,  label='Actual: Churned',  edgecolor='white')
    ax.axvline(0.5, color='gray', linestyle='--', linewidth=1.5, label='Decision boundary (0.5)')
    ax.set_xlabel('Predicted Probability of Churn')
    ax.set_ylabel('Count')
    ax.set_title(f'Probability Distribution — {name}')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('../outputs/eval_07_probability_dist.png', bbox_inches='tight')
plt.show()
print('Chart 07 saved ✓')

In [ ]:
# ── Final summary printout ─────────────────────────────────────────────────────
DIVIDER = '=' * 62
print(DIVIDER)
print('   PREDICTIVE MODELING — FINAL RESULTS SUMMARY')
print('   customer-churn-analysis / predictive-modeling')
print(DIVIDER)

print('\nMODEL PERFORMANCE')
print('-' * 62)
print(results_df[['Accuracy','Precision','Recall','F1','ROC-AUC']].round(4).to_string())

best = results_df['F1'].idxmax()
best_f1  = results_df.loc[best, 'F1']
best_auc = results_df.loc[best, 'ROC-AUC']
best_acc = results_df.loc[best, 'Accuracy']

print(f'\nBEST MODEL : {best}')
print(f'  Accuracy : {best_acc:.4f} ({best_acc*100:.1f}%)')
print(f'  F1 Score : {best_f1:.4f}')
print(f'  ROC-AUC  : {best_auc:.4f}')

print(f'\nTOP 5 PREDICTIVE FEATURES ({best})')
print('-' * 62)
for i, (_, row) in enumerate(feat_df.head(5).iterrows(), 1):
    print(f'  {i}. {row["Feature"]:<40} importance = {row["Importance"]:.4f}')

print(f'\nSAVED MODELS')
print('-' * 62)
print('  predictive-modeling/models/')
print('    logistic_regression.pkl')
print('    random_forest.pkl')
print('    random_forest_tuned.pkl')
print('    scaler.pkl')